# Wick's theorem, verified by brute forceExecutable companion to the Wick's theorem sections of chapter 3.A vacuum expectation value of a string of fermion creation and annihilationoperators can be computed two entirely different ways:1. **By anticommutation** — repeatedly apply   $\{a_p, a_q^\dagger\}=\delta_{pq}$ to push annihilation operators to the   right until they hit the vacuum. This is the elementary route of the   worked examples in the text, and it branches badly.2. **By Wick's theorem** — sum over all ways of pairing the operators into   contractions, with a sign for each crossing. Only the *fully contracted*   terms survive the vacuum expectation value.This notebook implements both and checks that they agree. That check *is* thecontent of the theorem.Operators are written as `(index, dagger)` pairs, so `('p', True)` is$a_p^\dagger$ and `('p', False)` is $a_p$. Indices stay symbolic, so resultscome out as sums of products of Kronecker deltas.

In [ ]:
from itertools import combinationsclass DeltaSum:    """A linear combination of products of Kronecker deltas."""    def __init__(self, terms=None):        self.terms = dict(terms) if terms else {}    @staticmethod    def zero():        return DeltaSum()    @staticmethod    def one():        return DeltaSum({(): 1})    @staticmethod    def delta(p, q):        if p == q:            return DeltaSum.one()                 # delta_pp = 1        return DeltaSum({((p, q) if p < q else (q, p),): 1})    def __add__(self, other):        out = dict(self.terms)        for term, coeff in other.terms.items():            out[term] = out.get(term, 0) + coeff            if out[term] == 0:                del out[term]        return DeltaSum(out)    def __mul__(self, other):        if isinstance(other, int):            return DeltaSum({t: c*other for t, c in self.terms.items()                             if c*other != 0})        out = {}        for t1, c1 in self.terms.items():            for t2, c2 in other.terms.items():                term = tuple(sorted(set(t1) | set(t2)))                out[term] = out.get(term, 0) + c1*c2                if out[term] == 0:                    del out[term]        return DeltaSum(out)    __rmul__ = __mul__    def __eq__(self, other):        return self.terms == other.terms    def __bool__(self):        return bool(self.terms)    def __repr__(self):        if not self.terms:            return "0"        pieces = []        for term, coeff in sorted(self.terms.items()):            body = "".join(f"d({p}{q})" for p, q in term) or "1"            sign = "+" if coeff > 0 else "-"            mag = "" if abs(coeff) == 1 else str(abs(coeff))            pieces.append(f"{sign} {mag}{body}")        text = " ".join(pieces)        return text[2:] if text.startswith("+ ") else text    def substitute(self, values):        total = 0        for term, coeff in self.terms.items():            product = 1            for p, q in term:                product *= 1 if values.get(p, p) == values.get(q, q) else 0                if product == 0:                    break            total += coeff*product        return totaldef c(index):    """A creation operator a_index^dagger."""    return (index, True)def a(index):    """An annihilation operator a_index."""    return (index, False)

## Route 1: anticommutationThe recursion is the one carried out by hand in the text. Find the leftmostplace where an annihilation operator stands immediately to the left of acreation operator and replace$$a_p a_q^\dagger \;\longrightarrow\; \delta_{pq} - a_q^\dagger a_p .$$When no such place remains the string is normal-ordered, and its vacuumexpectation value vanishes unless the string is empty.

In [ ]:
_call_count = [0]def vev_bruteforce(ops):    """<0| ops |0> by repeated use of the anticommutation relation."""    ops = tuple(ops)    _call_count[0] += 1    if len(ops) == 0:        return DeltaSum.one()    if len(ops) % 2 == 1:            # an odd string can never pair up        return DeltaSum.zero()    if ops[0][1]:                    # starts with a creation operator        return DeltaSum.zero()       #   -> kills the bra vacuum    if not ops[-1][1]:               # ends with an annihilation operator        return DeltaSum.zero()       #   -> kills the ket vacuum    for k in range(len(ops) - 1):        left, right = ops[k], ops[k+1]        if not left[1] and right[1]:                  # a_p a_q^dagger            rest = ops[:k] + ops[k+2:]            swapped = ops[:k] + (right, left) + ops[k+2:]            return (DeltaSum.delta(left[0], right[0]) * vev_bruteforce(rest)                    + vev_bruteforce(swapped) * (-1))    return DeltaSum.zero()           # normal-ordered and non-emptydef bruteforce_cost(ops):    _call_count[0] = 0    vev_bruteforce(ops)    return _call_count[0]

## Route 2: Wick's theoremOnly one of the four elementary contractions survives,$$\overset{\frown}{a_p a_q^\dagger} = \delta_{pq},\qquad\overset{\frown}{a_p a_q} =\overset{\frown}{a_q^\dagger a_p} =\overset{\frown}{a_p^\dagger a_q^\dagger} = 0 ,$$and each set of contractions carries the sign $(-1)^{\nu}$ with $\nu$ thenumber of crossings of the contraction lines.

In [ ]:
def contraction(x, y):    """<0| x y |0>: non-zero only for an annihilation operator to the left    of a creation operator."""    if (not x[1]) and y[1]:        return DeltaSum.delta(x[0], y[0])    return DeltaSum.zero()def perfect_matchings(indices):    """All ways of pairing up a list of positions."""    if not indices:        yield []        return    first, rest = indices[0], indices[1:]    for k in range(len(rest)):        remainder = rest[:k] + rest[k+1:]        for tail in perfect_matchings(remainder):            yield [(first, rest[k])] + taildef matching_sign(pairs):    """(-1) to the number of crossings of the contraction lines."""    crossings = 0    for (p, q), (r, s) in combinations(pairs, 2):        if p < r < q < s or r < p < s < q:            crossings += 1    return (-1)**crossingsdef vev_wick(ops):    """<0| ops |0> as a signed sum over fully contracted terms."""    ops = tuple(ops)    if len(ops) % 2 == 1:        return DeltaSum.zero()    total = DeltaSum.zero()    for pairs in perfect_matchings(list(range(len(ops)))):        value = DeltaSum.one()        for i, j in pairs:            value = value * contraction(ops[i], ops[j])            if not value:                break        if value:            total = total + value * matching_sign(pairs)    return totaldef double_factorial(m):    """(m-1)!! -- the number of perfect matchings of m objects."""    out, k = 1, m - 1    while k > 1:        out *= k        k -= 2    return out

## Two operatorsThe whole content of the theorem at $M=2$ is$xy = N[xy] + \overset{\frown}{xy}$, and only one of the four combinationshas a non-vanishing contraction.

In [ ]:
cases = (((a("p"), c("q")), "<0| a_p a_q^+ |0>"),         ((a("p"), a("q")), "<0| a_p a_q   |0>"),         ((c("p"), a("q")), "<0| a_p^+ a_q |0>"),         ((c("p"), c("q")), "<0| a_p^+ a_q^+ |0>"))for ops, label in cases:    brute, wick = vev_bruteforce(ops), vev_wick(ops)    print(f"{label:22s} = {str(brute):10s} (Wick: {str(wick):10s} "          f"agree: {brute == wick})")

## Three operatorsAn odd number of operators can never be paired up completely, so at least oneis always left uncontracted — and $\langle 0|a|0\rangle=\langle0|a^\dagger|0\rangle=0$.

In [ ]:
for ops, label in (((a("p"), c("q"), c("r")), "<0| a_p a_q^+ a_r^+ |0>"),                   ((a("p"), a("q"), c("r")), "<0| a_p a_q a_r^+   |0>"),                   ((c("p"), a("q"), c("r")), "<0| a_p^+ a_q a_r^+ |0>")):    brute, wick = vev_bruteforce(ops), vev_wick(ops)    print(f"{label:24s} = {brute}   (Wick: {wick}, agree: {brute == wick})")

## Four operators: the overlap $\langle rs|pq\rangle$Of the three ways to pair four operators, one vanishes on inspection, one isnested (no crossings, plus sign) and one crosses once (minus sign). The resultis the antisymmetrised overlap$$\langle rs|pq\rangle = \delta_{rp}\delta_{sq}-\delta_{sp}\delta_{rq}.$$

In [ ]:
ops = (a("s"), a("r"), c("p"), c("q"))brute, wick = vev_bruteforce(ops), vev_wick(ops)print("<0| a_s a_r a_p^+ a_q^+ |0>")print(f"  by anticommutation : {brute}")print(f"  by Wick's theorem  : {wick}")print(f"  agree              : {brute == wick}")print()print("the three pairings, one at a time:")labels = {((0, 1), (2, 3)): "(a_s,a_r)(a_p^+,a_q^+)  both contractions vanish",          ((0, 3), (1, 2)): "(a_s,a_q^+)(a_r,a_p^+)  nested, no crossing",          ((0, 2), (1, 3)): "(a_s,a_p^+)(a_r,a_q^+)  one crossing"}for pairs in perfect_matchings([0, 1, 2, 3]):    value = DeltaSum.one()    for i, j in pairs:        value = value * contraction(ops[i], ops[j])    key = tuple(sorted(tuple(sorted(p)) for p in pairs))    signed = value * matching_sign(pairs)    print(f"  {labels.get(key, str(key)):40s} -> {signed}")

## Six and eight operatorsThe agreement is not an accident of small numbers.

In [ ]:
alphabet = "abcdefghijklmnop"for m in (2, 4, 6, 8):    half = m // 2    names = alphabet[:m]    ops = tuple(a(n) for n in names[:half]) + tuple(c(n) for n in names[half:])    brute, wick = vev_bruteforce(ops), vev_wick(ops)    print(f"M = {m}: {len(brute.terms):3d} surviving terms, "          f"(M-1)!! = {double_factorial(m):3d} pairings, "          f"agree: {brute == wick}")

## The number operatorWith $|12\rangle = a_1^\dagger a_2^\dagger|0\rangle$ and$\hat N = \sum_i a_i^\dagger a_i$, Wick's theorem should return the particlenumber.

In [ ]:
total = 0for i in ("1", "2", "3", "4"):    ops = (a("2"), a("1"), c(i), a(i), c("1"), c("2"))    value = vev_wick(ops).substitute({})    total += value    print(f"  i = {i}: <12| a_{i}^+ a_{i} |12> = {value}")print(f"\n  sum over i = {total}   (= N, the particle number)")

## The two-body interaction$$\hat H_I=\tfrac12\sum_{pqrs}\langle pq|v|rs\rangle\,a_p^\dagger a_q^\dagger a_s a_r,\qquad|ij\rangle = a_i^\dagger a_j^\dagger|0\rangle .$$We evaluate the eight-operator string for every assignment of $(p,q,r,s)$ andsee which survive. The answer should be the antisymmetrised matrix element$\langle ij|v|ij\rangle - \langle ij|v|ji\rangle$.

In [ ]:
surviving = {}for p in ("i", "j"):    for q in ("i", "j"):        for r in ("i", "j"):            for s in ("i", "j"):                ops = (a("j"), a("i"), c(p), c(q), a(s), a(r), c("i"), c("j"))                value = vev_wick(ops).substitute({})                if value:                    surviving[(p, q, r, s)] = valuefor (p, q, r, s), value in sorted(surviving.items()):    print(f"  <{p}{q}|v|{r}{s}>  coefficient {value:+d}")direct = sum(v for k, v in surviving.items()             if k in (("i", "j", "i", "j"), ("j", "i", "j", "i")))exchange = sum(v for k, v in surviving.items()               if k in (("i", "j", "j", "i"), ("j", "i", "i", "j")))print()print(f"  with the factor 1/2:  {direct//2:+d} <ij|v|ij>  "      f"{exchange//2:+d} <ij|v|ji>")print("  so <ij|H_I|ij> = <ij|v|ij> - <ij|v|ji> = <ij|v|ij>_AS")

## Why the theorem is worth havingBoth routes grow quickly, but they grow differently. The anticommutationrecursion branches with no structure one can anticipate. Wick's theoremenumerates a known number of terms in advance, most of which vanish oninspection because one of their contractions is zero, and the survivors can bewritten down directly.

In [ ]:
print(f"{'M':>4s} {'pairings (M-1)!!':>18s} {'recursive calls':>17s} "      f"{'surviving terms':>17s}")for m in (2, 4, 6, 8, 10):    half = m // 2    names = alphabet[:m]    ops = tuple(a(n) for n in names[:half]) + tuple(c(n) for n in names[half:])    print(f"{m:4d} {double_factorial(m):18d} {bruteforce_cost(ops):17d} "          f"{len(vev_wick(ops).terms):17d}")

## Where this leadsEverything above concerns the *vacuum* expectation value. In many-body theorythe reference state is not the vacuum but a filled Fermi sea, and the usefulstatement is Wick's theorem relative to that reference — obtained by the sameargument once creation and annihilation operators are redefined with respectto the new vacuum, as in the particle-hole formalism.The generalised theorem, for a product of groups that are each alreadynormal-ordered, is what makes the calculations of perturbation theory andcoupled-cluster theory tractable: only contractions *between* different groupscontribute, which is a small fraction of the pairings counted above.